# SC-Mamba: Predictive Certainty Insights

This notebook visualizes the Stochastic outputs $\\mu$ (point prediction) and $\\sigma^2$ (variance) of SC-Mamba.

Unlike traditional point-forecast models (like original Mamba4Cast), our model optimizes the Spectral ELBO. We evaluate whether $\\sigma^2$ accurately balloons during periods of historical market volatility (news shocks, black swan drops).

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.append('../')
from core.models import SCMamba_Forecaster

### 1. Load Dummy Prediction DataFrame
*(During proper operation, load the `pred_df_...csv` exported from `eval_real_dataset_sc.py`)*

In [ ]:
PRED_LEN = 30

# Mocking a successful export from eval_real_dataset_sc.py
time_index = np.arange(PRED_LEN)
target_prices = np.sin(time_index * 0.2) * 10 + 150 # Mock sine-wave stock
pred_mu = target_prices + np.random.normal(0, 1.5, size=PRED_LEN)
# Injecting an artificial 'volatility spike' between t=15 and t=20
pred_sigma = np.ones(PRED_LEN) * 2.0 
pred_sigma[15:20] = 8.0 

mock_df = pd.DataFrame({
    'time': time_index,
    'target': target_prices,
    'pred_mu': pred_mu,
    'pred_sigma': pred_sigma
})

### 2. Plot Stochastic Confidence Intervals $\\pm 2\\sigma$

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(mock_df['time'], mock_df['target'], label="Ground Truth (NASDAQ Ticker)", color='black', linewidth=2)
plt.plot(mock_df['time'], mock_df['pred_mu'], label="SC-Mamba Prediction (\u03bc)", color='blue', linestyle='--')

# 95% Confidence Interval (2*Sigma)
lower_bound = mock_df['pred_mu'] - 2 * np.sqrt(mock_df['pred_sigma'])
upper_bound = mock_df['pred_mu'] + 2 * np.sqrt(mock_df['pred_sigma'])

plt.fill_between(mock_df['time'], lower_bound, upper_bound, color='blue', alpha=0.2, label="95% Confidence Bounds (\u00b12\u03c3)")

plt.title("SC-Mamba: Volatility Adaptive Risk Envelopes")
plt.xlabel("Time Step")
plt.ylabel("Asset Price")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()